## 1. Imports

In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms

import numpy as np
import os
from torch.utils.data import Dataset, DataLoader, random_split

import torch.nn as nn
import torchvision.models as models

## 2. Data Loading

In [11]:
import numpy as np
import torch
import os
from torch.utils.data import Dataset, DataLoader

# Custom dataset class to load .npy files
class NPYDataset(Dataset):
    def __init__(self, root):
        self.files = []
        self.classes = ['no', 'sphere', 'vort']
        
        for label, class_name in enumerate(self.classes):
            folder_path = os.path.join(root, class_name)
            for file_name in os.listdir(folder_path):
                if file_name.endswith('.npy'):
                    self.files.append((os.path.join(folder_path, file_name), label))
        
        print(f"Loaded {len(self.files)} images from {root}")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        file_path, label = self.files[idx]
        image = np.load(file_path)
        
        # Print shape once to understand the data
        if idx == 0:
            print(f"Image shape: {image.shape}")
        
        # Convert numpy to tensor
        image = torch.FloatTensor(image)
        
        # Add channel dimension if image is 2D
        if len(image.shape) == 2:
            image = image.unsqueeze(0)
        
        # Make sure image has 3 channels (repeat if needed)
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)
        elif image.shape[0] != 3:
            print(f"Warning: Unexpected shape {image.shape}")
        
        return image, label

# Paths
train_path = r'C:\Users\Admin\OneDrive - Sol Plaatje University\Desktop\Summer of Code\dataset\dataset\train'
val_path = r'C:\Users\Admin\OneDrive - Sol Plaatje University\Desktop\Summer of Code\dataset\dataset\val'

# Create datasets
train_dataset = NPYDataset(train_path)
val_dataset = NPYDataset(val_path)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"\nTraining batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Loaded 30000 images from C:\Users\Admin\OneDrive - Sol Plaatje University\Desktop\Summer of Code\dataset\dataset\train
Loaded 7500 images from C:\Users\Admin\OneDrive - Sol Plaatje University\Desktop\Summer of Code\dataset\dataset\val

Training batches: 938
Validation batches: 235


## 3. Model

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

import torch.nn as nn
import torchvision.models as models

# Using ResNet18 - a pretrained model that already understands basic image features
# This is called "transfer learning" and helps us train faster with less data
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # Load ResNet18 with pretrained weights (already knows edges, shapes, textures)
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # Freeze all layers (keep the pretrained knowledge)
        for param in self.resnet.parameters():
            param.requires_grad = False
        # Replace the final layer to output 3 classes (no, sphere, vort)
        # Original ResNet outputs 1000 classes, we only need 3
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(),              
            nn.Dropout(0.5),              
            nn.Linear(512, 3)             
        )
        
        # Only train the new layers we added
        for param in self.resnet.fc.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        # Pass image through the network and return predictions
        return self.resnet(x)

# Create the model and move it to GPU if available
model = Classifier().to(device)

# Loss function: measures how wrong the model is
criterion = nn.CrossEntropyLoss()

# Optimizer: updates the model to reduce the loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Model created successfully")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Using device: cpu
Model created successfully
Total parameters: 11,440,707
Trainable parameters: 264,195


## 4. Training the Model

In [13]:
# Training settings
num_epochs = 10  # Number of times the model will see all training images
best_val_acc = 0  # Track the best validation accuracy

# Lists to store metrics for plotting later
train_losses = []
val_losses = []
train_accs = []
val_accs = []

print("Starting training...\n")

#for epoch in range(num_epochs):
    # ========== TRAINING PHASE ==========
    model.train()  # Set model to training mode
    running_loss = 0
    correct = 0
    total = 0
    
    # Loop through all batches of training images
    for images, labels in train_loader:
        # Move data to device (CPU in your case)
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass: make predictions
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass: learn from mistakes
        optimizer.zero_grad()  # Reset gradients
        loss.backward()        # Calculate gradients
        optimizer.step()       # Update model weights
        
        # Track statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    # Calculate training metrics for this epoch
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # ========== VALIDATION PHASE ==========
    model.eval()  # Set model to evaluation mode
    running_loss = 0
    correct = 0
    total = 0
    all_labels = []
    all_probs = []
    
    # No gradients needed during validation (saves memory and speed)
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass only (no backward pass)
            outputs = model(images)
            loss = criterion(outputs, labels)
            probs = torch.softmax(outputs, dim=1)
            
            # Track statistics
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # Store for ROC curve later
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Calculate validation metrics for this epoch
    val_loss = running_loss / len(val_loader)
    val_acc = 100 * correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Print progress for this epoch
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    # Save model if it's the best so far
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  *** New best model saved! ***")
    
    print()

print(f"Training complete!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

Starting training...

Image shape: (1, 150, 150)
Image shape: (1, 150, 150)
Epoch 1/10
  Train Loss: 1.1059 | Train Acc: 34.51%
  Val Loss: 1.0985 | Val Acc: 36.27%
  *** New best model saved! ***

Image shape: (1, 150, 150)
Image shape: (1, 150, 150)
Epoch 2/10
  Train Loss: 1.0983 | Train Acc: 34.65%
  Val Loss: 1.1014 | Val Acc: 34.01%

Image shape: (1, 150, 150)
Image shape: (1, 150, 150)
Epoch 3/10
  Train Loss: 1.0969 | Train Acc: 35.12%
  Val Loss: 1.0932 | Val Acc: 36.64%
  *** New best model saved! ***

Image shape: (1, 150, 150)
Image shape: (1, 150, 150)
Epoch 4/10
  Train Loss: 1.0968 | Train Acc: 34.76%
  Val Loss: 1.0954 | Val Acc: 35.69%

Image shape: (1, 150, 150)
Image shape: (1, 150, 150)
Epoch 5/10
  Train Loss: 1.0953 | Train Acc: 35.20%
  Val Loss: 1.0916 | Val Acc: 37.29%
  *** New best model saved! ***

Image shape: (1, 150, 150)
Image shape: (1, 150, 150)
Epoch 6/10
  Train Loss: 1.0938 | Train Acc: 35.73%
  Val Loss: 1.0905 | Val Acc: 36.77%

Image shape: (1, 1

## 5. Evaluation

## 6. Results